In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif

df = pd.read_csv("00 customer.csv")

display(df.head())

,age,gender,review,education,purchased
0,30,Female,Average,School,No
1,68,Female,Poor,UG,No
2,70,Female,Good,PG,No
3,72,Female,Good,PG,No
4,16,Female,Average,UG,No


In [2]:
numeric_columns = df.select_dtypes(include=np.number).columns

print("Numeric Columns:")
print(numeric_columns)

Numeric Columns:
Index(['age'], dtype='object')


In [3]:
categorical_columns = df.select_dtypes(include="object").columns

print("Categorical Columns:")
print(categorical_columns)

Categorical Columns:
Index(['gender', 'review', 'education', 'purchased'], dtype='object')


In [4]:
X = df.drop("purchased", axis=1)

y = df["purchased"]

print("Input Features:")
display(X.head())

print("Target Column:")
display(y.head())

Input Features:


,age,gender,review,education
0,30,Female,Average,School
1,68,Female,Poor,UG
2,70,Female,Good,PG
3,72,Female,Good,PG
4,16,Female,Average,UG


Target Column:


,purchased
0,No
1,No
2,No
3,No
4,No


In [5]:
scaler = StandardScaler().set_output(transform="pandas")

numeric_cols = X.select_dtypes(include=["number"]).columns

scaled_data = scaler.fit_transform(X[numeric_cols])

display(scaled_data.head())

,age
0,-0.951170
1,0.544876
2,0.623615
3,0.702354
4,-1.502345


In [6]:
encoder = OneHotEncoder(
    sparse_output=False,
    drop="first"
).set_output(transform="pandas")

encoded_gender = encoder.fit_transform(X[["gender"]])

display(encoded_gender.head())

,gender_Male
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0


In [7]:
ordinal_encoder = OrdinalEncoder(
    categories=[
        ["Poor", "Average", "Good"],
        ["School", "UG", "PG"]
    ]
).set_output(transform="pandas")

encoded_ordinal = ordinal_encoder.fit_transform(
    X[["review", "education"]]
)

display(encoded_ordinal.head())

,review,education
0,1.0,0.0
1,0.0,1.0
2,2.0,2.0
3,2.0,2.0
4,1.0,1.0


In [8]:
X_final = pd.concat(
    [scaled_data, encoded_gender, encoded_ordinal],
    axis=1
)

display(X_final.head())

,age,gender_Male,review,education
0,-0.951170,0.0,1.0,0.0
1,0.544876,0.0,0.0,1.0
2,0.623615,0.0,2.0,2.0
3,0.702354,0.0,2.0,2.0
4,-1.502345,0.0,1.0,1.0


In [9]:
le = LabelEncoder()

y_encoded = le.fit_transform(y)

print("Original Target:")
print(y.head())

print("\nEncoded Target:")
print(y_encoded[:5])

Original Target:
0    No
1    No
2    No
3    No
4    No
Name: purchased, dtype: object

Encoded Target:
[0 0 0 0 0]


In [10]:
selector = SelectKBest(
    score_func=f_classif,
    k=3
).set_output(transform="pandas")

X_selected = selector.fit_transform(X_final, y_encoded)

print("Selected Features:")
display(X_selected.head())

print("Feature Scores:")
display(pd.DataFrame({
    "Feature": X_final.columns,
    "Score": selector.scores_
}).sort_values(by="Score", ascending=False))

Selected Features:


,age,gender_Male,education
0,-0.951170,0.0,0.0
1,0.544876,0.0,1.0
2,0.623615,0.0,2.0
3,0.702354,0.0,2.0
4,-1.502345,0.0,1.0


Feature Scores:


,Feature,Score
3,education,1.924906
1,gender_Male,0.002021
0,age,0.000003
2,review,0.000000
